In [7]:
import os
import pandas as pd
import PyPDF2

# Set the path to the folder containing PDF files
folder_path = '../../queplan_insurance'
pdf_files = [f for f in os.listdir(folder_path) if f.endswith('.pdf')]
df = pd.DataFrame(columns=['name', 'text'])

# Extract text from each PDF and create the dataset
for file_name in pdf_files:
    file_path = os.path.join(folder_path, file_name)
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        full_text = [pdf_reader.pages[page_num].extract_text() for page_num in range(len(pdf_reader.pages))]
        text = ' '.join(full_text)
        df = pd.concat([df, pd.DataFrame({'name': [file_name], 'text': [text]})], ignore_index=True)

df

,name,text
0,POL320180100.pdf,PÓLIZA DE SEGURO PARA PRESTACIONES MÉDICAS DER...
1,POL320190074.pdf,SEGURO PARA PRESTACIONES MÉDICAS DE ALTO COSTO...
2,POL320200214.pdf,SEGURO PARA PRESTACIONES MÉDICAS DE ALTO COSTO...
3,POL320210210.pdf,SEGURO PARA PRESTACIONES MÉDICAS DE ALTO COSTO...
4,POL320150503.pdf,PÓLIZA DE SEGURO PARA PRESTACIONES MÉDICAS DER...
5,POL320130223.pdf,SEGURO COLECTIVO COMPLEMENTARIO DE SALUD \nInc...
6,POL320210063.pdf,SEGURO INDIVIDUAL OBLIGATORIO DE SALUD ASOCIAD...
7,POL120190177.pdf,PÓLIZA DE ACCIDENTES PERSONALES / REEMBOLSO GA...
8,POL320200071.pdf,SEGURO INDIVIDUAL CATASTRÓFICO POR EVENTO\nInc...


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def load_data(filepath):
    """Load data from a CSV file."""
    return pd.read_csv(filepath)

def preprocess_themes(df):
    """Preprocess the 'Temas' column to handle multiple themes per policy."""
    df['Temas'] = df['Temas'].str.strip()
    df['Temas'] = df['Temas'].str.split('\n').apply(lambda x: [item.strip() for item in x if item.strip()])
    themes_flat = [theme for sublist in df['Temas'] for theme in sublist if theme]
    return pd.Series(themes_flat).value_counts().sort_values(ascending=False)

def preprocess_dates(df):
    """Extract and correct dates from the 'Fecha Depósito' field."""
    df['Fecha Depósito'] = df['Fecha Depósito'].str.extract(r'(\d{2}/\d{2}/\d{4})')
    df['Fecha Depósito'] = pd.to_datetime(df['Fecha Depósito'], format='%d/%m/%Y', errors='coerce')
    return df

def aggregate_data(df, date_column, freq):
    """Group data by the specified frequency."""
    return df.groupby(pd.Grouper(key=date_column, freq=freq)).size()

def plot_data(data, kind, title, xlabel, ylabel, folder, rotation=45, horizontal=False):
    """Generate and save a formatted bar chart."""
    plt.figure(figsize=(12, 8))
    ax = plt.subplot(111)
    if horizontal:
        # Sort values in ascending order for horizontal bar chart
        data.sort_values(ascending=True).plot(kind='barh', color='skyblue', ax=ax)
    else:
        # Sort values in descending order for vertical bar chart
        data.sort_values(ascending=False).plot(kind='bar', color='skyblue', ax=ax)
        # Format x-axis labels if the index is datetime type
        if pd.api.types.is_datetime64_any_dtype(data.index):
            ax.set_xticklabels([label.strftime('%Y') for label in data.index], rotation=rotation)
        else:
            plt.xticks(rotation=rotation)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.tight_layout()
    # Save the plot as a PNG file
    filepath = os.path.join(folder, f"{title.replace(' ', '_')}.png")
    plt.savefig(filepath)
    plt.close()

def main():
    filepath = 'cmf_policies.csv'
    output_folder = 'output_images'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Load data
    df = load_data(filepath)
    
    # Process themes and plot
    theme_counts = preprocess_themes(df)
    plot_data(theme_counts, 'barh', 'Number of Policies per Theme', 'Number of Policies', 'Themes', output_folder, horizontal=True)
    
    # Process dates and plot
    df = preprocess_dates(df)
    annual_counts = aggregate_data(df, 'Fecha Depósito', 'YE')
    plot_data(annual_counts, 'bar', 'Number of Policies Deposited by Year', 'Year', 'Number of Policies', output_folder)
    
    # Process entity counts and plot top 20 entities
    entity_counts = df['Nombre entidad que deposita'].value_counts().head(20).sort_values(ascending=False)
    plot_data(entity_counts, 'barh', 'Number of Policies Deposited by Top 20 Entities', 'Number of Policies', 'Entity Name', output_folder, horizontal=True)
    
    # Process 'codigo' prefixes and plot
    df['codigo_prefix'] = df['Código'].str[:3]
    prefix_counts = df['codigo_prefix'].value_counts()
    plot_data(prefix_counts, 'barh', 'Count of each type of document', 'Count', 'Prefix', output_folder, horizontal=True)

if __name__ == '__main__':
    main()
